# Webcam test for the bean classifier
Run the cells in order with **Shift+Enter**. Put this notebook in the same folder as `bean_classifier.keras`.

In [ ]:
# 1. Install OpenCV (only needed once; skip if already installed)
%pip install opencv-python

In [ ]:
# 2. Load the saved model
import cv2, numpy as np, time
import tensorflow as tf
import matplotlib.pyplot as plt

model = tf.keras.models.load_model("bean_classifier.keras")
IMG_SIZE = (224, 224)
class_names = ["bad", "good"]   # alphabetical folder order: 0 = bad, 1 = good
print("Model loaded")

In [ ]:
# 3. Take ONE photo from the webcam and classify it
cam = cv2.VideoCapture(0)
time.sleep(2)
for _ in range(5):
    cam.read()
ok, frame = cam.read()
cam.release()

if not ok:
    print("Camera not found")
else:
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    small = cv2.resize(rgb, IMG_SIZE)
    arr = np.expand_dims(small.astype("float32"), 0)
    p = float(model.predict(arr, verbose=0)[0][0])
    label = class_names[int(p > 0.5)]
    conf = p if p > 0.5 else 1 - p
    plt.imshow(rgb); plt.axis("off")
    plt.title(f"{label} ({conf:.0%})"); plt.show()

In [ ]:
# 4. LIVE view (smoother): the video runs every frame, but the model only
#    classifies every 5th frame, so the feed doesn't lag.
#    Click the camera window and press q to close it.
cam = cv2.VideoCapture(0)
frame_count = 0
text = "..."
while True:
    ok, frame = cam.read()
    if not ok:
        print("Camera not found")
        break
    if frame_count % 5 == 0:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        small = cv2.resize(rgb, IMG_SIZE)
        arr = np.expand_dims(small.astype("float32"), 0)
        p = float(model.predict(arr, verbose=0)[0][0])
        label = class_names[int(p > 0.5)]
        conf = p if p > 0.5 else 1 - p
        text = f"{label} {conf:.0%}"
    frame_count += 1
    cv2.putText(frame, text, (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3)
    cv2.imshow("Bean classifier (press q to quit)", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
cam.release()
cv2.destroyAllWindows()